# SDE cascade

**Difficulty:** Intermediate

Extract with a cheap model, verify the extraction with a `Noul` battery, escalate to an expensive reasoning model only when a verifier signal actually fires. Most of a big model's quality, at a fraction of its cost — because the expensive model only ever sees the extractions that need it.

Ask your agent if anything's unclear.

<details>
  <summary><strong>Primary source (trimmed):</strong> docs.typesafe.ai/cookbooks/sde_cascade (click to expand)</summary>

# SDE cascade

> Uses a 2-stage structured-data-extraction cascade (mini -> verify -> reasoning) to get most of the quality of a big reasoning model at a fraction of the cost.

Big reasoning models extract structured data well but are slow and expensive; small models are cheap but make mistakes. A cascade gets most of the quality at a fraction of the cost:

1. **Extract** with a cheap/small model.
2. **Verify** with TypeSafe: a per-field `Noul` battery, each question framed so `true` = something is wrong.
3. **Escalate** to an expensive reasoning model only if a verifier signal fires; otherwise keep the cheap answer.

## The real worked example: a page with nothing to extract

An NYU events-calendar page, scraped, asked for two fields: `registration_open_date` and `description` ("a brief description of the registration open date"). The page is nav and boilerplate only — no registration date, no description anywhere in it. A well-behaved extractor should decline to invent either field.

The cheap model (`gpt-5.4-mini`) didn't decline. It returned `registration_open_date: ""` (correctly empty) but confidently fabricated `description: "Registration opens for the fall semester"` — schema-valid, plausible-sounding, and entirely made up.

## The verifier battery, and what it found

Per field, per metric, one `Noul` each (`hallucinated`, `off_target`, `unreasonable`, `name_desc_mismatch`, `type_mismatch`, `incomplete`, `format_violation` for filled fields; just `absence_wrong` for empty ones) — all in one request:

```
qid                                      P(wrong)
--------------------------------------------------
description::hallucinated                    0.95  <== FIRES
description::off_target                      0.85  <== FIRES
description::unreasonable                     0.58
registration_open_date::absence_wrong         0.14
description::format_violation                 0.10
description::name_desc_mismatch               0.08
description::type_mismatch                    0.02
```

The verifier concentrated the signal exactly on the field that's actually wrong — high on `description` (fabricated), low on `registration_open_date` (correctly empty).

## The gate: `any_flag`, not a mean

Escalate if *any* field's flag exceeds the threshold (`0.7` here) — a `max`-style gate, not an average, so one confident red flag is enough instead of being averaged into silence by six quiet fields. Here it fires on `description::hallucinated` (0.95) and `description::off_target` (0.85), so the record escalates to the reasoning model:

```
mini      : {"registration_open_date": "", "description": "Registration opens for the fall semester"}
reasoning : {"description": "", "registration_open_date": ""}
```

The reasoning model drops the fabricated description entirely — it recognized the page never describes a registration date, and declined to invent one. The cascade turned a confident, schema-valid fabrication into an honest empty field, and only spent reasoning-model dollars on this one field *because the verifier said to*.

## Across 100 prompts

The cascade's cost/quality frontier sits above and to the left of every single model run alone — sweeping the escalation threshold buys most of the top reasoning model's quality for a fraction of its cost, because the cheap rung handles easy items for nearly free and only flagged items pay for the expensive one.

> The full cookbook runs a 7-metric battery per non-empty field plus a holistic `__overall__::judge` head (shown but not used for gating), against a real scraped 100k-row dataset, and plots the full cost/quality Pareto frontier across 100 prompts — trimmed here to the two gating metrics on the real worked example. Read the full cookbook at [docs.typesafe.ai/cookbooks/sde_cascade](https://docs.typesafe.ai/cookbooks/sde_cascade) any time.

</details>

## The core idea

Confidence-gated routing (lesson 7) applied to a *second model's output* instead of a live decision: the cheap extraction is the thing being routed, `Noul` is the verifier, and the gate decides whether to trust the cheap answer or pay for a better one. The `any_flag` (max, not mean) detail matters — one clearly-wrong field among several correct ones would get diluted to a moderate average and might not fire a threshold at all; taking the *worst* signal instead means a single confident hallucination is never hidden behind five quiet correct fields.

In [ ]:
import { client } from "../../assets/typesafe-client.ts";
import { noul } from "npm:@typesafe-ai/sdk@^0.7.0";

const schema = {
  registration_open_date:
    "The date registration opens for the fall semester, mm/dd/yyyy, or blank if unsure.",
  description: "A brief description of the registration open date.",
};

// A short stand-in for the real scraped page: nav and boilerplate, no registration info at all.
const sourceExcerpt =
  "Events Calendar - New York University. Search Events. About the Events Calendar. " +
  "Events Calendar Tutorial. Info for Students, Faculty, Alumni, Employees, Community. " +
  "Equal Opportunity and Non-Discrimination at NYU. Unless otherwise noted, all content " +
  "copyright New York University. All rights reserved.";

// The cheap model's real extraction from the cookbook's own run: correctly empty on the
// date, but confidently fabricated on the description.
const miniExtraction = {
  registration_open_date: "",
  description: "Registration opens for the fall semester",
};

const response = await client.systemOne({
  state: {
    schema,
    sourceText: sourceExcerpt,
    extraction: miniExtraction,
  },
  questions: {
    descriptionHallucinated: noul(
      "Is the `description` field unsupported by, or absent from, the source text?",
      {
        true:
          "The description is a hallucination - not supported by, or absent from, the source text.",
        false: "The description is supported by the source text.",
      },
    ),
    descriptionOffTarget: noul(
      "Does the source text fail to genuinely report what the `description` field describes, so the value was pulled from incidental text?",
      {
        true:
          "The source does not genuinely provide this field - the value was pulled from incidental text.",
        false: "The source genuinely reports this field.",
      },
    ),
    registrationDateAbsenceWrong: noul(
      "The `registration_open_date` field is empty. Does the source text actually contain a registration date, making the empty result wrong?",
      {
        true: "A value was wrongly omitted.",
        false: "Returning nothing is correct.",
      },
    ),
  },
});

const { descriptionHallucinated, descriptionOffTarget, registrationDateAbsenceWrong } =
  response.answers;

const FIRE_THRESHOLD = 0.7;
const checks: Record<string, number> = {
  "description::hallucinated": descriptionHallucinated.noul,
  "description::off_target": descriptionOffTarget.noul,
  "registration_open_date::absence_wrong": registrationDateAbsenceWrong.noul,
};
const fired = Object.entries(checks).filter(([, p]) => p > FIRE_THRESHOLD);

console.log("checks:", checks);
console.log(
  fired.length > 0
    ? `ESCALATE (fired: ${fired.map(([id]) => id).join(", ")})`
    : "ACCEPT cheap result",
);

## What just happened

The exact fabrication the cookbook's own `gpt-5.4-mini` run produced, verified by our own three-question battery instead of the full seven-metric one. Expect `descriptionHallucinated` and `descriptionOffTarget` both high (the source genuinely says nothing about registration) and `registrationDateAbsenceWrong` low (leaving that field empty was the correct call) — which should fire the gate and correctly flag this record for escalation.

**Try it yourself:** change `miniExtraction.description` to an honest empty string (what the reasoning model actually returned) and re-run. Predict whether `descriptionHallucinated` and `descriptionOffTarget` both drop low enough that the gate now says "accept" instead of "escalate" — the verifier should recognize an honest non-answer as correct, not just a confident one as wrong.

## Related

- `lessons/patterns/0007-confidence-gated-routing.ipynb` — the same escalate-or-accept shape, applied to a second model's output instead of a live user-facing decision
- `cookbooks/how-to/classifying-rag-passages.ipynb` — another `Noul` battery read by thresholds in code, though there the thresholds pick a route rather than a max-gate

Questions? Ask your agent — it has the same research this cookbook was built from, in `research/typesafe-ai.md`.